# ema-first-moment — ex2: second-moment EMA update v = beta2*v + (1-beta2)*g**2

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ema-first-moment`. Running the final beacon cell reports progress against the `Optimizer: Adam EMA first moment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA first moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-first-moment`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-first-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA first moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## EMAs of `g` vs `g**2` side-by-side — quick refresher

Adam carries TWO EMAs. ex1 covered the first moment `m = beta1*m + (1-beta1)*g`. The SECOND moment is structurally identical but EMAs `g**2`:
```
v_t = beta2 * v_{t-1} + (1 - beta2) * g_t**2
```
Key differences from `m`:
- `v` is ALWAYS non-negative (squared input).
- Default `beta2 = 0.999` (vs beta1=0.9) — longer averaging window (~1000 steps) because we want a stable variance estimate.
- `v` does NOT preserve sign — it captures gradient MAGNITUDE only.

```python
for m_buf, v_buf, g in zip(m_list, v_list, grads):
    m_buf.copy_(beta1 * m_buf + (1 - beta1) * g)
    v_buf.copy_(beta2 * v_buf + (1 - beta2) * g.pow(2))
```

### Exercise 2 — second-moment EMA update v = beta2*v + (1-beta2)*g**2

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Adam second-moment recurrence `v = beta2*v + (1-beta2)*g**2` via `buffer.copy_()` so the variance-EMA buffer stays non-negative across steps regardless of gradient sign.
> Keywords: adam, second-moment, ema, squared-gradient
> ```

**KCs targeted:** `ema-second-moment-recurrence-squared-g`, `buffer-copy_-mutates-state-in-place`

Implement `ex2_ema_v_step(v_list, grad_list, beta2)`. The second-moment update from Adam.

For each `(v, g)` in `zip(v_list, grad_list)`:

1. Compute `beta2 * v + (1 - beta2) * g.pow(2)`.
2. Mutate `v` in place: `v.copy_(...)`. Don't rebind.
3. Append `v` to the return list (by reference).

Inputs:
- `v_list`: list of per-param second-moment buffers (mutated).
- `grad_list`: list of per-param gradients (NOT mutated).
- `beta2`: float in `(0, 1)` — Adam default is `0.999`.

Output: list of updated `v` tensors.

**Critical invariant:** `v` must be NON-NEGATIVE elementwise at every step — the squaring guarantees this. The test verifies with deliberately-negative gradients.

In [ ]:
def ex2_ema_v_step(v_list, grad_list, beta2):
    out = []
    for v, g in zip(v_list, grad_list):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
        out.append(v)
    return out


<details><summary>Solution</summary>

```python
def ex2_ema_v_step(v_list, grad_list, beta2):
    out = []
    for v, g in zip(v_list, grad_list):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
        out.append(v)
    return out
```

**Why a separate drill for the second moment.** Structurally it's the first-moment update with `g.pow(2)` instead of `g` — same recurrence shape, same in-place semantics. But conflating the two (e.g. `v = beta1*v + ...` with the wrong beta, or forgetting to square) is the #2 source of Adam bugs after the rebind. Practicing the second-moment fold on its own builds the muscle memory for `g.pow(2)`.

**`g.pow(2)` vs `g * g` vs `g ** 2`.** All three are equivalent; `g.pow(2)` is what PyTorch's reference Adam uses (it's a single dispatched op rather than two). For autograd this matters very little; for clarity the choice is taste.

**Connecting to Adam's update.** The full Adam step is `theta -= lr * m_hat / (sqrt(v_hat) + eps)`. The `sqrt(v_hat)` is why `v` must stay non-negative — `sqrt` of a negative would be NaN, and the optimizer would silently produce all-NaN parameters within one step.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()